# 002_meeting_neutral_tables — 中立層四張表＋三個核心 View（DDL）

- 用途：建會議萃取中立層的表與 View（規格 §3.0b、§3.1、§3.2、§3.3、§3.6；STATUS.md §6.1 加的欄）。只建結構、不灌資料；資料由 W1 / W2 定期 Job 寫入。
- **正本是 [c04] `table_specs` 與 [c05] `view_specs`**（欄名、型別、COMMENT 都在這裡，隨 notebook 進 git）。改欄 = 開新流水號 migration 做 `ALTER TABLE ADD COLUMNS`，不改本 notebook 的表（§2.3 只增不改）；View 無狀態，改 [c05] 重跑即可。
- 冪等：表 `CREATE TABLE IF NOT EXISTS`；View `CREATE OR REPLACE VIEW`。重跑無害；[c30] 會比對實際欄位與 spec，缺欄時印出 `ALTER TABLE` 語句但不自動執行。
- 規格 §11.1 延後的欄（`sub_no`、`title_hint`、`dept_name`、`time_refs` / `numbers` 結構化…）**先開欄、不填值**，加欄便宜、改鍵貴。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證 spec 與印出 DDL，不建任何物件）
- 負責人 / 更新日期：（填）/ 2026-10-15

| 物件 | 鍵 / 來源 | 內容 | 敏感度 |
|---|---|---|---|
| `s_meeting_record_mail_bu` | `pk_seq` | 一列一封信的可讀版：來源選擇、初洗後 `body_display`、轉寄人拆欄、mail 層去重 `body_hash / dup_of_pk` | **S3**（含人名、email、原文） |
| `s_meeting_record_clean_bu` | `(pk_seq, segment_no, sub_no)` | 一列一段：token 化 `clean_text`、`display_start / end` 偏移、segment 層去重 `content_hash / dup_of` | S2 |
| `s_meeting_participants_bu` | `(pk_seq, segment_no, token, role)` | 一列一個「人 × 角色 × 段」；token ↔ 真實身分的唯一對照 | **S3** |
| `g_meeting_items_bu` | `item_id` | 一列一條原子陳述；`item_key` 穩定鍵、`supersedes_item_id` 重萃承接 | S0–S2 |
| `v_meeting_items_q` | items ⋈ participants | 有效條目（`ok` 且未 superseded）＋陣列欄字串版＋發言人公司 / 部門；不含姓名 email | S2 |
| `v_meeting_item_models` | explode(`model_nos`) | `(item_id, model_no)` 橋表 | S2 |
| `v_meeting_mail_index` | mail ⋈ clean ⋈ participants ⋈ items | 一列一封信的列表頁：段數、條目數、客戶眾數、參與公司；不含正文、不含人名 | S1 |
| `b_meeting_migration_log` | `(slug, run_at)` | migration 執行紀錄（001 已建，這裡 IF NOT EXISTS 以防跑在新 schema） | – |

**不在本 migration**：column mask / GRANT（§3.6 `v_meeting_participants_q` 延後）、分應用 View、Sales Brief 鏡頭兩張表（§3.4、§3.5，等 W1 跑順再開 003）。

In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run})

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c05] 不碰 spark / dbutils，可被 tests/ 載入。
import json
import re

SLUG = "002_meeting_neutral_tables"
UNIT = "bu"  # 單位代碼：表名後綴與 unit 欄預設值（規格 §3 命名規則）
COL_RE = r"^[a-z][a-z0-9_]*$"
TYPE_RE = r"^(STRING|BIGINT|INT|DOUBLE|BOOLEAN|DATE|TIMESTAMP|ARRAY<STRING>)$"

# STATUS.md §6.1 / 規格 §11.1「不能省的三件事」：事後補等於全量重跑，DDL 一定要有。
REQUIRED_COLS = {
    "s_meeting_record_mail_bu": ["body_hash", "dup_of_pk", "has_headers"],
    "s_meeting_record_clean_bu": ["content_hash", "dup_of", "display_start", "display_end"],
    "g_meeting_items_bu": ["item_key", "supersedes_item_id"],
}
# S3 欄位：只准落在 s_meeting_record_mail_bu / s_meeting_participants_bu；任何 View 不得暴露。
S3_COLS = {
    "display_name_raw", "name_zh", "name_en", "email", "emp_id", "account",
    "forwarder_name_zh", "forwarder_emp_id", "forwarder_account",
    "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display",
}

In [ ]:
# [c03] pure_ddl
# spec → SQL 字串的純函式；不碰 spark。測試在本機用同一份 spec 建空表跑 View 的 SELECT。
def q(s: str | None) -> str:
    """SQL 單引號字串常值；None → ''。"""
    return "'" + (s or "").replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_create_table(fq: str, spec: dict) -> str:
    """CREATE TABLE IF NOT EXISTS；鍵欄 NOT NULL；有 cluster_by 才加 CLUSTER BY（liquid clustering）。"""
    keys = set(spec["key"])
    cols = ",\n".join(
        f"    {name} {typ}{' NOT NULL' if name in keys else ''} COMMENT {q(comment)}"
        for name, typ, comment in spec["columns"]
    )
    cluster = f"\nCLUSTER BY ({', '.join(spec['cluster_by'])})" if spec.get("cluster_by") else ""
    return f"CREATE TABLE IF NOT EXISTS {fq} (\n{cols}\n) USING DELTA{cluster}\nCOMMENT {q(spec['comment'])}"


def build_select(spec: dict, fq) -> str:
    """View 的查詢本體：spec['select'] = [(expr, alias, comment)]，spec['body'](fq) 回傳 FROM 起的 SQL。
    fq(table_name) -> 可查詢的名稱；正式是三段式，測試用 temp view 名。"""
    select = ",\n".join(f"    {expr} AS {alias}" for expr, alias, _ in spec["select"])
    return f"SELECT\n{select}\n{spec['body'](fq)}"


def build_create_view(fq_view: str, spec: dict, fq) -> str:
    """CREATE OR REPLACE VIEW，欄位清單帶 COMMENT（Genie / 人看的語意）。"""
    col_list = ",\n".join(f"    {alias} COMMENT {q(comment)}" for _, alias, comment in spec["select"])
    return (
        f"CREATE OR REPLACE VIEW {fq_view} (\n{col_list}\n)\nCOMMENT {q(spec['comment'])}\nAS\n"
        + build_select(spec, fq)
    )


def schema_ddl(spec: dict) -> str:
    """欄位 → 'a BIGINT, b STRING'（測試建空表、[c30] 比對型別用）。"""
    return ", ".join(f"{n} {t}" for n, t, _ in spec["columns"])


def add_columns_sql(fq: str, spec: dict, missing: list[str]) -> str:
    """[c30] 發現缺欄時印出的補欄語句（不自動執行；要補就開新流水號 migration）。"""
    by_name = {n: (t, c) for n, t, c in spec["columns"]}
    parts = ", ".join(f"{n} {by_name[n][0]} COMMENT {q(by_name[n][1])}" for n in missing)
    return f"ALTER TABLE {fq} ADD COLUMNS ({parts})"


def validate_specs(tables: dict, views: dict) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。"""
    errs = []
    for t, spec in tables.items():
        names = [c[0] for c in spec["columns"]]
        dup = sorted({n for n in names if names.count(n) > 1})
        if dup:
            errs.append(f"{t} 欄位重複：{dup}")
        for n, typ, comment in spec["columns"]:
            if not re.match(COL_RE, n):
                errs.append(f"{t}.{n} 欄名格式不符（小寫英數底線）")
            if not re.match(TYPE_RE, typ):
                errs.append(f"{t}.{n} 型別不在允許清單：{typ}")
            if not comment:
                errs.append(f"{t}.{n} 沒有 COMMENT")
        for k in list(spec["key"]) + list(spec.get("cluster_by", [])):
            if k not in names:
                errs.append(f"{t} key / cluster_by 欄 {k} 不存在")
        for r in REQUIRED_COLS.get(t, []):
            if r not in names:
                errs.append(f"{t} 缺必要欄 {r}")
        if not spec.get("comment"):
            errs.append(f"{t} 沒有表 COMMENT")
    for v, spec in views.items():
        aliases = [a for _, a, _ in spec["select"]]
        dup = sorted({a for a in aliases if aliases.count(a) > 1})
        if dup:
            errs.append(f"{v} 輸出欄重複：{dup}")
        leak = sorted(S3_COLS & set(aliases))
        if leak:
            errs.append(f"{v} 暴露 S3 欄：{leak}")
        for _, a, c in spec["select"]:
            if not re.match(COL_RE, a):
                errs.append(f"{v}.{a} 欄名格式不符")
            if not c:
                errs.append(f"{v}.{a} 沒有 COMMENT")
        if not spec.get("comment"):
            errs.append(f"{v} 沒有 View COMMENT")
    return errs

In [ ]:
# [c04] table_specs
# 正本。欄位 = (名稱, 型別, COMMENT)。鍵欄自動 NOT NULL。延後欄（§11.1）照開、W1 先填 NULL。
# CLUSTER BY (pk_seq)：四張表都以 pk_seq 回溯 / MERGE，日量百封級，不做 partition。
_SRC = "同 Bronze b_internal_meeting_record"

TABLES = {
    "s_meeting_record_mail_bu": {
        "comment": "會議紀錄信展示表（一列一封信，幾乎等於 Bronze 的可讀版）。S3：含真實人名、email、原文數字，權限同 Bronze，不掛 Genie。正本：002_meeting_neutral_tables [c04]。",
        "key": ["pk_seq"],
        "cluster_by": ["pk_seq"],
        "columns": [
            ("pk_seq", "BIGINT", "主鍵，" + _SRC),
            ("drupal_id", "STRING", "回溯用，" + _SRC),
            ("drupal_link", "STRING", "回溯連結，多為 NULL；NULL 時以 pk_seq 顯示"),
            ("unit", "STRING", "單位代碼（本版一律 bu，D4：整表都算 BU）"),
            ("create_date", "TIMESTAMP", "入庫時間；W1 增量水位"),
            ("mail_date", "DATE", "信件日期，" + _SRC),
            ("mail_datetime", "TIMESTAMP", "信件時間，" + _SRC),
            ("subject_raw", "STRING", "原主旨"),
            ("subject_clean", "STRING", "去 FW / RE 前綴後的主旨"),
            ("forwarder_emp_id", "STRING", "轉寄人工號（mail_from 第 1 段）"),
            ("forwarder_name_zh", "STRING", "轉寄人中文姓名（mail_from 第 2 段）"),
            ("forwarder_account", "STRING", "轉寄人帳號（mail_from 第 3 段）"),
            ("forwarder_dept_code", "STRING", "轉寄人部門代碼（mail_from 第 4 段，例 DTDDB0）"),
            ("forwarder_dept_name", "STRING", "部門名稱；待決 #13 無對照表前為 NULL"),
            ("forwarder_token", "STRING", "轉寄人在 s_meeting_participants_bu 的 token（例 P01）；轉寄人不是會議參與者"),
            ("mail_to_raw", "STRING", "頂層收件人原字串（Phase 0 全空）"),
            ("mail_cc_raw", "STRING", "頂層副本原字串（Phase 0 全空）"),
            ("mail_bcc_raw", "STRING", "頂層密件原字串（Phase 0 全空）"),
            ("meeting_when", "TIMESTAMP", "由正文 **When:** 解析的會議時間（pk 層級）"),
            ("meeting_where", "STRING", "由正文 **Where:** 解析的地點"),
            ("body_source_field", "STRING", "正文來源欄：mail_body_md / mail_without_fw / mail_body / none"),
            ("has_headers", "BOOLEAN", "正文是否含 From / Sent 標頭；mail_without_fw 來源一律 false，此時參與者只有轉寄人"),
            ("body_conflict", "BOOLEAN", "候選正文之間講不同會議（Phase 1 起抽樣，未抽到為 NULL）"),
            ("body_display", "STRING", "初洗後正文（刪 base64 圖片、圖片佔位、Word 殘渣、免責聲明；連結縮成文字），保留人名 email 與標頭，markdown 可直接渲染"),
            ("raw_len", "INT", "初洗前長度（字元）"),
            ("display_len", "INT", "初洗後長度（字元）"),
            ("is_fw", "BOOLEAN", "轉寄旗標，" + _SRC),
            ("attach_count", "INT", "附件數（來源 has_attach）"),
            ("oversize", "BOOLEAN", "raw_len > 200k（T1-a 關卡，仍入表但不分段）"),
            ("body_hash", "STRING", "mail 層去重鍵：sha1(normalize(body_display))；normalize = 去空白、全半形統一、去標點"),
            ("dup_of_pk", "BIGINT", "body_hash 命中較早 pk_seq 時填其值（整封重複，Phase 0 為 23%）；首次出現 NULL。非 NULL 的信不分段不萃取"),
            ("cleaner_version", "STRING", "初洗規則版本（與 clean 表共用版本號）"),
            ("cleaned_at", "TIMESTAMP", "寫入時間"),
        ],
    },
    "s_meeting_record_clean_bu": {
        "comment": "會議紀錄清洗分段表（一列一段：thread 的一則回覆或紀要的一個 ITEM 區塊）。輸入是展示表 body_display。clean_text 已 token 化去識別，是 quote 驗證的唯一基準。不進 Genie。正本：002_meeting_neutral_tables [c04]。",
        "key": ["pk_seq", "segment_no", "sub_no"],
        "cluster_by": ["pk_seq"],
        "columns": [
            ("pk_seq", "BIGINT", "來源主鍵"),
            ("segment_no", "INT", "段序；0 = 整筆未分段"),
            ("sub_no", "INT", "超長段再切的子序，預設 0（§11.1 延後：Phase 1 不切，超長標記跳過）"),
            ("drupal_id", "STRING", "回溯用"),
            ("drupal_link", "STRING", "回溯連結，多為 NULL"),
            ("unit", "STRING", "單位代碼（本版 bu）"),
            ("create_date", "TIMESTAMP", "入庫時間（水位）"),
            ("mail_date", "DATE", "信件日期"),
            ("mail_datetime", "TIMESTAMP", "信件時間"),
            ("segment_date", "DATE", "本段實際日期：thread 取 **Sent:**，紀要取 ITEM 標頭日期"),
            ("date_inferred", "BOOLEAN", "取不到段日期、以 mail_date 代替時為 true"),
            ("subject_raw", "STRING", "原主旨"),
            ("subject_clean", "STRING", "去 FW / RE 前綴後的主旨"),
            ("forwarder_token", "STRING", "轉寄人 token；不是會議參與者，下游不得當發言者"),
            ("forwarder_dept_code", "STRING", "轉寄人部門代碼（單位判定、權限稽核用）"),
            ("meeting_when", "TIMESTAMP", "pk 層級會議時間，複製到每段"),
            ("meeting_where", "STRING", "pk 層級會議地點，複製到每段"),
            ("parties", "ARRAY<STRING>", "本段出現的組織網域（auo.com、acer.com），供發言方判定"),
            ("customer_hint", "STRING", "程式推測的客戶，只給 L1 參考、不當事實"),
            ("app_hint", "STRING", "程式推測的應用（TV / NB / MNT），只給 L1 參考"),
            ("content_type", "STRING", "inline_minutes / thread_reply / appointment / attachment_only / noise"),
            ("body_source_field", "STRING", "正文來源欄（複製自展示表）"),
            ("body_conflict", "BOOLEAN", "候選正文衝突旗標（複製自展示表）"),
            ("display_start", "INT", "本段在展示表 body_display 的起始字元偏移（0-based，含）"),
            ("display_end", "INT", "本段在展示表 body_display 的結束字元偏移（不含）"),
            ("clean_text", "STRING", "清洗、token 化去識別（<P01>）、表格還原後的正文；quote 驗證的唯一基準"),
            ("raw_len", "INT", "本段清洗前長度"),
            ("clean_len", "INT", "本段清洗後長度（token 預算用）"),
            ("is_fw", "BOOLEAN", "轉寄旗標"),
            ("attach_count", "INT", "附件數"),
            ("content_hash", "STRING", "segment 層去重鍵：sha1(normalize(clean_text))，token 化之後才算，不同轉寄人也能命中"),
            ("dup_of", "STRING", "命中既有段時填其 pk_seq-segment_no-sub_no；首次出現 NULL。非 NULL 的段不送 T2"),
            ("cleaner_version", "STRING", "清洗規則版本"),
            ("cleaned_at", "TIMESTAMP", "寫入時間"),
        ],
    },
    "s_meeting_participants_bu": {
        "comment": "會議紀錄參與者表（一列一個人或群組 × 角色 × 段；同一人在同一 pk 內共用一個 token）。S3：token 與真實身分的唯一對照；外部人員資訊只存這裡，不進 repo。正本：002_meeting_neutral_tables [c04]。",
        "key": ["pk_seq", "segment_no", "token", "role"],
        "cluster_by": ["pk_seq"],
        "columns": [
            ("pk_seq", "BIGINT", "來源主鍵"),
            ("segment_no", "INT", "出現在哪一段的標頭；頂層 mail_from / to / cc / bcc 為 0"),
            ("token", "STRING", "pk 內唯一：人 P01、P02…；群組 G01…；clean_text 中以 <P01> 出現"),
            ("role", "STRING", "forwarder / from / to / cc / bcc / signature"),
            ("kind", "STRING", "person / group（distribution list）"),
            ("display_name_raw", "STRING", "標頭原字串（例 Allen KC Huang 黃國誠、Wang, Example）"),
            ("name_zh", "STRING", "拆出的中文名"),
            ("name_en", "STRING", "拆出的英文名"),
            ("email", "STRING", "可為 NULL（md 標頭有時只留顯示名）"),
            ("email_domain", "STRING", "由 email 取，小寫；無 email 為 NULL"),
            ("side", "STRING", "auo / customer / third_party / unknown；由 email_domain 查 s_semantic_entity_alias（後綴比對）"),
            ("company", "STRING", "s_semantic_entity.entity_id；對不上 NULL"),
            ("emp_id", "STRING", "AUO 工號（只有 mail_from 格式帶）"),
            ("account", "STRING", "AUO 帳號（同上）"),
            ("dept_code", "STRING", "AUO 部門代碼（例 DTDDB0）；外部人員 NULL"),
            ("dept_name", "STRING", "部門名稱；待決 #13 無對照表前為 NULL"),
            ("title_hint", "STRING", "簽名檔職稱，低信心；§11.1 延後，Phase 1 為 NULL"),
            ("resolver_version", "STRING", "參與者解析規則版本"),
            ("resolved_at", "TIMESTAMP", "寫入時間"),
        ],
    },
    "g_meeting_items_bu": {
        "comment": "會議紀錄原子條目（一列一條陳述：一個對象＋一個主張；L1 萃取）。人以 token 指涉，不得出現姓名 email。消費端請用 v_meeting_items_q。正本：002_meeting_neutral_tables [c04]。",
        "key": ["item_id"],
        "cluster_by": ["pk_seq"],
        "columns": [
            ("item_id", "STRING", "BU-MTG-<pk_seq>-<segment_no>-<sub_no>-<nnn>；每次萃取新發"),
            ("item_key", "STRING", "穩定鍵：sha1(pk_seq ‖ segment_no ‖ normalize(quote))；重萃後同一句話不變，鏡頭表以此承接審核欄"),
            ("supersedes_item_id", "STRING", "重萃時命中相同 item_key 的舊條目 id；首次 NULL"),
            ("pk_seq", "BIGINT", "回溯：來源信"),
            ("segment_no", "INT", "回溯：段序"),
            ("sub_no", "INT", "回溯：子段序"),
            ("unit", "STRING", "單位代碼（本版 bu）"),
            ("subject_clean", "STRING", "會議名稱（複製自清洗表，讓條目自帶來源）"),
            ("statement_type", "STRING", "fact / plan / request / decision / issue / opinion / definition（§6.1）"),
            ("speaker_token", "STRING", "L1 判定的發言人 token（例 P03）；紀要格式判不出為 NULL"),
            ("speaker_side", "STRING", "auo / customer / third_party / unknown；程式由 speaker_token 查參與者表，token 為 NULL 才採 L1 判斷"),
            ("customer_raw", "STRING", "素材原名（VD、海信、acer）"),
            ("customer", "STRING", "經 s_semantic_entity_alias 正規化的 entity_id；對不上 NULL"),
            ("app", "STRING", "TV / NB / MNT / ALL / NULL"),
            ("app_basis", "STRING", "model_prefix / spec_context / customer_default / subject / unknown（§6.4）"),
            ("model_nos", "ARRAY<STRING>", "提及的型號，可多個；橋表見 v_meeting_item_models"),
            ("topic_tags", "ARRAY<STRING>", "受控主題標籤 1–3 個（§6.2 詞表）"),
            ("topic_tag_primary", "STRING", "topic_tags[0]，最能代表主張的標籤；Genie 與 facet 規則首選"),
            ("free_tags", "ARRAY<STRING>", "自由關鍵字（AG25、LVDS…）"),
            ("summary", "STRING", "一句乾淨版，繁中，型號原文；人以 token 指涉，不得還原姓名"),
            ("quote", "STRING", "逐字原文（清洗後），必須是 clean_text 子字串"),
            ("event_date", "DATE", "陳述描述的事發生或被說的日期（§6.5）"),
            ("event_month", "STRING", "yyyy-MM；Sales Brief 鏡頭的 brief_month 由此推"),
            ("date_basis", "STRING", "segment_header / inline_date / mail_date_fallback"),
            ("time_refs", "STRING", "JSON 字串 [{when, what}]；§11.1 延後結構化驗證，先存原樣。顯示用，不做篩選"),
            ("numbers", "STRING", "JSON 字串 [{value, unit, what}]，單位原樣；先存原樣。顯示用，不做篩選"),
            ("terms_mentioned", "ARRAY<STRING>", "命中詞彙表的術語（BP、限簽、Allocation…）"),
            ("sensitivity", "STRING", "S0 / S1 / S2，程式判（§6.6）；S3 不應出現"),
            ("confidence", "DOUBLE", "LLM 自評 0–1，只做排序與抽審"),
            ("extract_status", "STRING", "ok / failed_schema / failed_quote"),
            ("superseded", "BOOLEAN", "重萃後舊條目標 true，不刪"),
            ("prompt_version", "STRING", "lineage：L1 prompt 版本"),
            ("model", "STRING", "lineage：模型名"),
            ("cleaner_version", "STRING", "lineage：對應的 clean_text 版本"),
            ("extracted_at", "TIMESTAMP", "寫入時間"),
        ],
    },
    "b_meeting_migration_log": {
        "comment": "會議萃取 migration 執行紀錄（只增不改）",
        "key": ["slug", "run_at"],
        "columns": [
            ("slug", "STRING", "migration 流水號＋名稱（例 002_meeting_neutral_tables）"),
            ("run_at", "TIMESTAMP", "執行時間"),
            ("run_by", "STRING", "current_user()"),
            ("result", "STRING", "JSON：各物件 created / exists / replaced"),
            ("note", "STRING", "備註"),
        ],
    },
}

In [ ]:
# [c05] view_specs
# 正本。select = [(expr, 輸出欄名, COMMENT)]；body(fq) 回傳 FROM 起的 SQL，fq(表名) 給完整名稱。
# 原則（§3.6）：消費者不直接掛原表；只留有效列；陣列欄另給逗號字串版；任何 View 不出現姓名 / email。
_ACTIVE = "i.extract_status = 'ok' AND i.superseded = false"


def _items_body(fq):
    # 發言人補公司 / 部門：同一 token 在 pk 內可能多列（多段、多角色），取 segment_no / role 最小的一列。
    return f"""FROM {fq('g_meeting_items_bu')} i
LEFT JOIN (
    SELECT pk_seq, token, company, side, dept_code
    FROM (
        SELECT pk_seq, token, company, side, dept_code,
               ROW_NUMBER() OVER (PARTITION BY pk_seq, token ORDER BY segment_no, role) AS rn
        FROM {fq('s_meeting_participants_bu')}
    )
    WHERE rn = 1
) p ON p.pk_seq = i.pk_seq AND p.token = i.speaker_token
WHERE {_ACTIVE}"""


def _models_body(fq):
    return f"""FROM {fq('g_meeting_items_bu')} i
LATERAL VIEW EXPLODE(i.model_nos) m AS model_no
WHERE {_ACTIVE}"""


def _mail_index_body(fq):
    return f"""FROM {fq('s_meeting_record_mail_bu')} m
LEFT JOIN (
    SELECT pk_seq, SUM(n) AS segment_count, SUM(n_dup) AS dup_segment_count,
           array_join(array_sort(collect_list(concat(coalesce(content_type, 'unknown'), ':', n))), ',') AS content_types
    FROM (
        SELECT pk_seq, content_type, COUNT(*) AS n,
               SUM(CASE WHEN dup_of IS NOT NULL THEN 1 ELSE 0 END) AS n_dup
        FROM {fq('s_meeting_record_clean_bu')}
        GROUP BY pk_seq, content_type
    )
    GROUP BY pk_seq
) c ON c.pk_seq = m.pk_seq
LEFT JOIN (
    SELECT pk_seq, COUNT(*) AS item_count, mode(customer) AS customer, mode(app) AS app,
           MAX(sensitivity) AS sensitivity_max
    FROM {fq('g_meeting_items_bu')} i
    WHERE {_ACTIVE}
    GROUP BY pk_seq
) i ON i.pk_seq = m.pk_seq
LEFT JOIN (
    SELECT pk_seq, array_sort(collect_set(company)) AS participant_companies
    FROM {fq('s_meeting_participants_bu')}
    WHERE side IN ('customer', 'third_party') AND company IS NOT NULL
    GROUP BY pk_seq
) p ON p.pk_seq = m.pk_seq"""


VIEWS = {
    "v_meeting_items_q": {
        "comment": "會議條目（BU）消費介面：只含有效條目（extract_status = ok 且未 superseded），陣列欄附逗號字串版，發言人補公司 / 部門代碼。不含姓名與 email。Genie / 語意萃取用。",
        "body": _items_body,
        "select": [
            ("i.item_id", "item_id", "條目 id"),
            ("i.item_key", "item_key", "穩定鍵（重萃不變）"),
            ("i.pk_seq", "pk_seq", "來源信 pk_seq"),
            ("i.segment_no", "segment_no", "來源段序"),
            ("i.sub_no", "sub_no", "來源子段序"),
            ("i.unit", "unit", "單位代碼"),
            ("i.subject_clean", "subject_clean", "會議名稱（主旨去 FW / RE）"),
            ("i.statement_type", "statement_type", "陳述類型：fact / plan / request / decision / issue / opinion / definition"),
            ("i.speaker_token", "speaker_token", "發言人 token（P03）；紀要判不出為 NULL"),
            ("COALESCE(i.speaker_side, p.side)", "speaker_side", "發言方：auo / customer / third_party / unknown"),
            ("p.company", "speaker_company", "發言人公司（entity_id）；由 token 查參與者表"),
            ("p.dept_code", "speaker_dept_code", "發言人 AUO 部門代碼；外部人員 NULL"),
            ("i.customer_raw", "customer_raw", "素材原名"),
            ("i.customer", "customer", "正規化客戶（entity_id）"),
            ("i.app", "app", "應用：TV / NB / MNT / ALL"),
            ("i.app_basis", "app_basis", "app 推導依據"),
            ("i.model_nos", "model_nos", "型號陣列（篩選請用 v_meeting_item_models）"),
            ("array_join(i.model_nos, ', ')", "model_nos_str", "型號，逗號字串"),
            ("i.topic_tags", "topic_tags", "受控主題標籤陣列"),
            ("i.topic_tag_primary", "topic_tag_primary", "主標籤"),
            ("array_join(i.topic_tags, ', ')", "topic_tags_str", "主題標籤，逗號字串"),
            ("i.free_tags", "free_tags", "自由關鍵字陣列"),
            ("array_join(i.free_tags, ', ')", "free_tags_str", "自由關鍵字，逗號字串"),
            ("i.summary", "summary", "一句摘要（繁中，人以 token 指涉）"),
            ("i.quote", "quote", "逐字原文（已 token 化去識別）"),
            ("i.event_date", "event_date", "事件日期"),
            ("i.event_month", "event_month", "事件月份 yyyy-MM"),
            ("i.date_basis", "date_basis", "日期依據"),
            ("i.time_refs", "time_refs", "未來時程 JSON；顯示用，不做篩選"),
            ("i.numbers", "numbers", "數字 JSON；顯示用，不做篩選"),
            ("i.terms_mentioned", "terms_mentioned", "命中術語陣列"),
            ("array_join(i.terms_mentioned, ', ')", "terms_str", "命中術語，逗號字串"),
            ("i.sensitivity", "sensitivity", "敏感度 S0 / S1 / S2"),
            ("i.confidence", "confidence", "LLM 自評信心 0–1"),
            ("i.prompt_version", "prompt_version", "lineage：prompt 版本"),
            ("i.model", "model", "lineage：模型"),
            ("i.cleaner_version", "cleaner_version", "lineage：清洗版本"),
            ("i.extracted_at", "extracted_at", "萃取時間"),
        ],
    },
    "v_meeting_item_models": {
        "comment": "會議條目 × 型號橋表（BU）：一列一個 (item_id, model_no)，讓「某型號的所有陳述」能 join。只含有效條目。",
        "body": _models_body,
        "select": [
            ("i.item_id", "item_id", "條目 id"),
            ("m.model_no", "model_no", "型號（原文）"),
            ("i.pk_seq", "pk_seq", "來源信 pk_seq"),
            ("i.customer", "customer", "正規化客戶"),
            ("i.app", "app", "應用"),
            ("i.event_date", "event_date", "事件日期"),
            ("i.topic_tag_primary", "topic_tag_primary", "主標籤"),
        ],
    },
    "v_meeting_mail_index": {
        "comment": "會議紀錄信列表頁（BU）：一列一封信，附段數、條目數、客戶眾數、參與公司。不含正文、不含人名。原文閱讀器（方向 D）用。",
        "body": _mail_index_body,
        "select": [
            ("m.pk_seq", "pk_seq", "信件主鍵"),
            ("m.drupal_id", "drupal_id", "回溯 id"),
            ("m.drupal_link", "drupal_link", "回溯連結，多為 NULL"),
            ("m.unit", "unit", "單位代碼"),
            ("m.mail_date", "mail_date", "信件日期"),
            ("m.mail_datetime", "mail_datetime", "信件時間"),
            ("m.create_date", "create_date", "入庫時間"),
            ("m.subject_clean", "subject_clean", "主旨（去 FW / RE）"),
            ("m.forwarder_dept_code", "forwarder_dept_code", "轉寄人部門代碼"),
            ("m.forwarder_dept_name", "forwarder_dept_name", "轉寄人部門名稱（待對照表）"),
            ("m.meeting_when", "meeting_when", "會議時間（有 Appointment 才有）"),
            ("m.meeting_where", "meeting_where", "會議地點"),
            ("m.body_source_field", "body_source_field", "正文來源欄"),
            ("m.has_headers", "has_headers", "正文是否含 From / Sent 標頭"),
            ("m.body_conflict", "body_conflict", "候選正文衝突"),
            ("m.oversize", "oversize", "超大信（不分段）"),
            ("m.dup_of_pk", "dup_of_pk", "整封重複時指向較早的 pk_seq"),
            ("m.display_len", "display_len", "初洗後正文長度"),
            ("m.attach_count", "attach_count", "附件數"),
            ("COALESCE(c.segment_count, 0)", "segment_count", "段數"),
            ("COALESCE(c.dup_segment_count, 0)", "dup_segment_count", "其中跨信重複的段數"),
            ("c.content_types", "content_types", "各 content_type 段數，例 inline_minutes:3,noise:1"),
            ("COALESCE(i.item_count, 0)", "item_count", "有效條目數"),
            ("i.customer", "customer", "條目客戶眾數（entity_id）"),
            ("i.app", "app", "條目應用眾數"),
            ("i.sensitivity_max", "sensitivity_max", "條目最高敏感級"),
            ("p.participant_companies", "participant_companies", "外部參與公司（entity_id 陣列，不含 AUO）"),
            ("m.cleaner_version", "cleaner_version", "初洗規則版本"),
        ],
    },
}

In [ ]:
# [c10] build_ddl
errors = validate_specs(TABLES, VIEWS)
for e in errors:
    print("ERROR", e)
assert not errors, f"spec 驗證失敗 {len(errors)} 筆，修 [c04] / [c05] 後重跑"


def fq(name: str) -> str:
    return f"{catalog}.{schema}.{name}"


table_ddl = {t: build_create_table(fq(t), s) for t, s in TABLES.items()}
view_ddl = {v: build_create_view(fq(v), s, fq) for v, s in VIEWS.items()}
existing = {t: spark.catalog.tableExists(fq(t)) for t in list(TABLES) + list(VIEWS)}
print("既有物件：", {k: v for k, v in existing.items() if v} or "無")
print("將建立：", [k for k, v in existing.items() if not v])
if dry_run:
    print("dry_run = true：只印 DDL，不建任何物件。確認後把 dry_run 改 false 重跑。\n")
    for ddl in list(table_ddl.values()) + list(view_ddl.values()):
        print(ddl, end=";\n\n")

In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。既有表的欄位差異由 [c30] 報告，不在這裡 ALTER。
if not dry_run:
    for t, ddl in table_ddl.items():
        spark.sql(ddl)
        print(("exists " if existing[t] else "created") + " " + fq(t))

In [ ]:
# [c21] create_views
# View 無狀態：CREATE OR REPLACE，改 [c05] 後重跑即生效。必須在 [c20] 之後（View 引用四張表）。
if not dry_run:
    for v, ddl in view_ddl.items():
        spark.sql(ddl)
        print(("replaced" if existing[v] else "created ") + " " + fq(v))

In [ ]:
# [c22] write_log
# 每次正式執行（dry_run = false）寫一列 b_meeting_migration_log；只 INSERT，不改舊列。
if not dry_run:
    result = {
        "tables": {t: ("exists" if existing[t] else "created") for t in TABLES},
        "views": {v: ("replaced" if existing[v] else "created") for v in VIEWS},
    }
    spark.sql(
        f"INSERT INTO {fq('b_meeting_migration_log')} "
        "SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result, ensure_ascii=False), "note": None},
    )
    print("logged", result)

In [ ]:
# [c30] check_target
# 比對實際表欄位 vs spec：缺欄（表早已存在、spec 後來加欄）→ 印補欄語句並 fail；型別不同 → 警告。
# View：能查、輸出欄 = spec、不含 S3 欄。
if not dry_run:
    problems = []
    for t, spec in TABLES.items():
        actual = {f.name: f.dataType.simpleString() for f in spark.table(fq(t)).schema.fields}
        missing = [n for n, _, _ in spec["columns"] if n not in actual]
        if missing:
            problems.append(f"{t} 缺欄 {missing}。補欄（開新流水號執行）：\n  {add_columns_sql(fq(t), spec, missing)}")
        for n, typ, _ in spec["columns"]:
            if n in actual and actual[n] != typ.lower():
                print(f"WARN {t}.{n} 型別 spec={typ.lower()} actual={actual[n]}")
        extra = [n for n in actual if n not in {c[0] for c in spec["columns"]}]
        if extra:
            print(f"INFO {t} 有 spec 以外的欄（不處理）：{extra}")
    for v, spec in VIEWS.items():
        cols = spark.table(fq(v)).columns
        want = [a for _, a, _ in spec["select"]]
        if cols != want:
            problems.append(f"{v} 輸出欄與 spec 不同：{cols}")
        leak = sorted(S3_COLS & set(cols))
        if leak:
            problems.append(f"{v} 暴露 S3 欄：{leak}")
        spark.table(fq(v)).limit(1).collect()  # 能執行（表空也要能跑）
    for p in problems:
        print("PROBLEM", p)
    assert not problems, f"{len(problems)} 個問題"
    print("tables:", {t: len(spark.table(fq(t)).columns) for t in TABLES})
    print("views :", {v: len(spark.table(fq(v)).columns) for v in VIEWS})